In [1]:
import sys
sys.path.insert(0, "/root/crypto-research/common")
sys.path.insert(0, "/root/crypto-research")

In [2]:
from factor_system import data_provider as dpmod
from factor_system.factor_hub.main import FactorManager

H5_PATH = "/root/crypto-research/common/data/raw/h5/bybit_linear_1m_unified_gp_research.h5"
BASE_DIR = "/root/crypto-research/common/factor_base"

# Register raw fields that exist in the unified GP H5 but are not in the default
# factor_system DataProvider alias table. Run this before creating FactorManager.
dpmod.DATASET_ALIASES.update({
    "xbinance_quote_volume": "xbinance_quote_volume",
    "xbinance_taker_buy_quote_volume": "xbinance_taker_buy_quote_volume",
    "xbinance_taker_buy_volume": "xbinance_taker_buy_volume",
    "xbinance_trade_count": "xbinance_trade_count",
})

fm = FactorManager(
    h5_path=H5_PATH,
    base_dir=BASE_DIR,
)


In [ ]:
# 评估示例因子（默认 lookback 365 天、8h 调仓）
# 这里演示显式指定区间；如果不传 params，则默认回看最近 365 天。
alpha_path = "/root/crypto-research/users/wesleywu/alphas/unsubmit/crypto_alpha_gp_residual_std_typical_price_money_flow_dvranklow03_w180_lag1.py"
result = fm.evaluate(alpha_path,profile_id="perp_1d",plot=True,params={"start":"2022-01-01","end":"2026-08-10","cost":0.0015})

In [4]:
import copy

import numpy as np
import pandas as pd

if "result" not in globals():
    raise ValueError("Run fm.evaluate(...) first so result exists before this cell.")

profile_id = globals().get("profile_id", "perp_1d")
start = globals().get("start", "2022-01-01")
end = globals().get("end", "2026-05-01")
cost = globals().get("cost", 0.0015)
periods_per_year = globals().get("periods_per_year", 365)

print("返回 keys:", list(result.keys()))
print()

print("factor_value shape:", result["factor_value"].shape)
print(
    "factor_value 时间范围:",
    result["factor_value"].index[0],
    "->",
    result["factor_value"].index[-1],
)
print()

print("全样本绩效(raw):")
perf = result["factor_performance"]["raw"]
for label, key in [
    ("RankIC", "rankic"),
    ("RankICIR", "rankicir"),
    ("ls_netir", "ls_netir"),
    ("ls_netret_annualized", "ls_netret"),
    ("turnover", "ls_turnover"),
    ("coverage", "coverage"),
    ("max drawdown", "ls_netmaxdd"),
]:
    v = perf.get(key, None)
    print(f"  {label}: {v:.4f}" if isinstance(v, float) else f"  {label}: {v}")
print()


def normalize_factor_result_index(factor_result, start, profile_id):
    fr = copy.deepcopy(factor_result)

    sample = None
    for key in ["long_rets", "short_rets", "rankics", "long_turnovers", "short_turnovers"]:
        if key in fr:
            sample = fr[key]
            break
    if sample is None:
        raise ValueError(f"factor_result available keys: {list(fr.keys())}")

    if isinstance(sample.index, pd.DatetimeIndex):
        return fr

    freq_map = {"perp_1d": "1D", "perp_8h": "8H", "perp_4h": "4H"}
    dt_freq = freq_map.get(profile_id, "1D")
    dt_index = pd.date_range(start=pd.Timestamp(start), periods=len(sample), freq=dt_freq)

    for key in [
        "long_rets",
        "long_turnovers",
        "long_nums",
        "short_rets",
        "short_turnovers",
        "short_nums",
        "rankics",
        "coverages",
        "universe_nums",
    ]:
        if key in fr and isinstance(fr[key], pd.Series) and len(fr[key]) == len(dt_index):
            fr[key].index = dt_index

    if "group_rets" in fr and isinstance(fr["group_rets"], pd.DataFrame) and len(fr["group_rets"]) == len(dt_index):
        fr["group_rets"].index = dt_index
    return fr


def calc_yearly_ls_metrics(factor_result, start, end, cost, periods_per_year):
    """Yearly long-short metrics using the same cost convention as factor_system."""
    long_rets = factor_result["long_rets"].loc[start:end]
    short_rets = factor_result["short_rets"].loc[start:end]
    long_to = factor_result["long_turnovers"].loc[start:end]
    short_to = factor_result["short_turnovers"].loc[start:end]

    # factor_system convention:
    # long_net = long_ret - long_turnover / 2 * cost
    # short_net = short_ret - short_turnover / 2 * cost
    # ls_net = (long_net + short_net) / 2
    long_netrets = long_rets - long_to / 2 * cost
    short_netrets = short_rets - short_to / 2 * cost

    # short_rets is already inverted by FactorResultEngine, so add it directly.
    ls_rets = (long_rets + short_rets) / 2
    ls_netrets = (long_netrets + short_netrets) / 2
    ls_turnover = (long_to + short_to) / 2

    rows = []
    for year, year_netrets in ls_netrets.groupby(ls_netrets.index.year):
        idx = year_netrets.index
        nav = (1 + year_netrets).cumprod()
        dd = nav / nav.cummax() - 1

        sharpe = np.nan
        if len(year_netrets) >= 2 and year_netrets.std() > 1e-12:
            sharpe = year_netrets.mean() / year_netrets.std() * np.sqrt(periods_per_year)

        rows.append({
            "period": str(year),
            "ls_netret": float((1 + year_netrets).prod() - 1),
            "ls_netmaxdd": float(-dd.min()) if len(dd) else np.nan,
            "ls_net_sharpe": float(sharpe) if np.isfinite(sharpe) else np.nan,
            "ls_win_rate": float((ls_rets.loc[idx] > 0).mean()) if len(idx) else np.nan,
            "ls_net_win_rate": float((year_netrets > 0).mean()) if len(idx) else np.nan,
            "ls_turnover": float(ls_turnover.loc[idx].mean()) if len(idx) else np.nan,
            "n_bars": int(len(year_netrets)),
            "is_full_year": bool(len(year_netrets) >= periods_per_year * 0.9),
        })

    return pd.DataFrame(rows).set_index("period")


def calc_full_sample_cum_ls_netret(factor_result, start, end, cost):
    long_rets = factor_result["long_rets"].loc[start:end]
    short_rets = factor_result["short_rets"].loc[start:end]
    long_to = factor_result["long_turnovers"].loc[start:end]
    short_to = factor_result["short_turnovers"].loc[start:end]

    long_netrets = long_rets - long_to / 2 * cost
    short_netrets = short_rets - short_to / 2 * cost
    ls_netrets = (long_netrets + short_netrets) / 2
    return float((1 + ls_netrets).prod() - 1)


raw_factor_result = result["factor_result"]["raw"]
fr = normalize_factor_result_index(raw_factor_result, start=start, profile_id=profile_id)

yearly = calc_yearly_ls_metrics(
    fr,
    start=pd.Timestamp(start),
    end=pd.Timestamp(end),
    cost=cost,
    periods_per_year=periods_per_year,
)

yearly_chain = float((1 + yearly["ls_netret"]).prod() - 1)
full_cum = calc_full_sample_cum_ls_netret(fr, pd.Timestamp(start), pd.Timestamp(end), cost)

print(f"自洽校验: 年度复利={yearly_chain:.4f}, 全样本累计={full_cum:.4f}, diff={yearly_chain - full_cum:+.6f}")
print(f"说明: perf['ls_netret']={perf['ls_netret']:.4f} 是年化收益，不应和年度复利直接比较。")
print()

print("按年绩效:")
print(yearly.to_string(float_format=lambda x: f"{x:.4f}"))
print()

print("仅完整年份:")
full_year_cols = ["ls_netret", "ls_netmaxdd", "ls_net_sharpe", "ls_win_rate", "ls_turnover"]
print(yearly[yearly["is_full_year"]][full_year_cols].to_string(float_format=lambda x: f"{x:.4f}"))

返回 keys: ['id', 'factor_value', 'factor_result', 'factor_performance']

factor_value shape: (2422081, 539)
factor_value 时间范围: 2022-01-01 00:00:00 -> 2026-08-10 00:00:00

全样本绩效(raw):
  RankIC: 0.0194
  RankICIR: 3.3963
  ls_netir: 1.1169
  ls_netret_annualized: 0.1770
  turnover: 0.0516
  coverage: 0.9287
  max drawdown: 0.1972

自洽校验: 年度复利=1.1811, 全样本累计=1.1811, diff=+0.000000
说明: perf['ls_netret']=0.1770 是年化收益，不应和年度复利直接比较。

按年绩效:
        ls_netret  ls_netmaxdd  ls_net_sharpe  ls_win_rate  ls_net_win_rate  ls_turnover  n_bars  is_full_year
period                                                                                                        
2022      -0.0280       0.1194        -0.1276       0.3479           0.3452       0.0295     365          True
2023       0.1592       0.1148         0.8778       0.5068           0.5068       0.0527     365          True
2024       0.3174       0.0686         1.9599       0.5301           0.5273       0.0507     366          True
2025       0

In [ ]:
# Copy this into a Jupyter cell.
# It assumes you either already have result from fm.evaluate(...),
# or you set alpha_path below and let this cell run fm.evaluate first.

import sys
from pathlib import Path

import pandas as pd

from use import h5_path

sys.path.insert(0, "/home/wesleywu/barra")
sys.path.insert(0, "/root/crypto-research/common")

from alphagen.tests.test_alpha2_mcts_smoke import H5
from crypto_barra_exposure import BarraConfig, analyze_evaluate_result
from factor_system import data_provider as dpmod
from factor_system.factor_hub.main import FactorManager

H5_PATH = "/root/crypto-research/common/data/raw/h5/bybit_linear_1m_unified_gp_research.h5"
BASE_DIR = "/root/crypto-research/common/factor_base"

dpmod.DATASET_ALIASES.update({
    "xbinance_quote_volume": "xbinance_quote_volume",
    "xbinance_taker_buy_quote_volume": "xbinance_taker_buy_quote_volume",
    "xbinance_taker_buy_volume": "xbinance_taker_buy_volume",
    "xbinance_trade_count": "xbinance_trade_count",
})

def ensure_result():
    if "result" in globals():
        return globals()["result"]
    if alpha_path is None:
        raise ValueError("No result found. Set alpha_path or run fm.evaluate(...) before this cell.")

    fm = FactorManager(
        h5_path=H5_PATH,
        base_dir=BASE_DIR,
    )
    return fm.evaluate(
        alpha_path,
        profile_id="perp_1d",
        plot=True,
        params={"start": "2022-01-01", "end": "2026-05-01", "cost": 0.0015},
    )


result = ensure_result()

cfg = BarraConfig(
    profile_id="perp_1d",
    signal_freq="1D",
    min_count=80,
    winsor_q=0.01,
)

barra = analyze_evaluate_result(
    result,
    h5_path=h5_path,
    cfg=cfg,
)

style_summary = barra["style_summary"]
daily_corr = barra["daily_style_corr"]
daily_reg = barra["daily_barra_regression"]
alpha_resid = barra["alpha_barra_residual"]

print("Barra style correlation summary:")
display(
    style_summary[
        [
            "style",
            "days",
            "pearson_mean",
            "pearson_abs_mean",
            "spearman_mean",
            "spearman_abs_mean",
            "beta_mean",
            "beta_t",
        ]
    ].sort_values("pearson_abs_mean", ascending=False)
)

print("Daily Barra regression R2:")
display(daily_reg[["date", "n", "r2"]].tail())
print("mean_r2 =", daily_reg["r2"].mean())

print("Daily Pearson correlation pivot summary:")
corr_pivot = daily_corr.pivot(index="date", columns="style", values="pearson")
display(
    corr_pivot.describe().T.sort_values("mean", key=lambda s: s.abs(), ascending=False)
)

print("Most recent day style correlations:")
last_date = daily_corr["date"].max()
display(
    daily_corr[daily_corr["date"] == last_date]
    .sort_values("pearson", key=lambda s: s.abs(), ascending=False)
)

print("Residual alpha panel:", alpha_resid.shape)
display(alpha_resid.tail())


In [4]:
import pandas as pd
from factor_system.data_provider import DataProvider
from factor_system.factor_hub.core.factor_result_profiles import PROFILES

def print_latest_groups(result, fm, profile_id="perp_1d", use_plot_group_lag=False):
    profile = PROFILES[profile_id]
    freq = profile["signal_freq"]

    fv = result["factor_value"]

    # 和 FactorResultEngine 一致：先按 profile 的 signal_freq/signal_method 重采样
    signal = DataProvider.fast_resample(fv, freq, profile["signal_method"])

    # 当前目标分组用 delay_bars；图里的 group_rets 用 delay_bars + 1
    lag = int(profile["delay_bars"]) + (1 if use_plot_group_lag else 0)
    signal = signal.shift(lag)

    start = signal.index.min()
    end = signal.index.max()
    universe = fm.dp.get_resampled_data(
        profile["universe"],
        freq,
        profile["universe_method"],
        start=start,
        end=end,
    ).fillna(False).astype(bool)

    signal, universe = signal.align(universe, join="inner", axis=None)
    signal = signal.where(universe)

    latest_ts = signal.dropna(how="all").index[-1]
    latest = signal.loc[latest_ts].dropna()

    rank_pct = latest.rank(pct=True, ascending=True)

    group = pd.Series(index=latest.index, dtype="object")
    for g in range(10):
        lo = g / 10.0
        hi = (g + 1) / 10.0 if g != 9 else 1.0 + 1e-9
        group[(rank_pct >= lo) & (rank_pct < hi)] = f"group_{g}"

    qt = float(profile["qt"])
    side = pd.Series("middle", index=latest.index)
    side[rank_pct >= 1.0 - qt] = "long"
    side[rank_pct <= qt] = "short"

    out = pd.DataFrame({
        "factor": latest,
        "rank_pct": rank_pct,
        "group": group,
        "side": side,
    }).sort_values("rank_pct", ascending=False)

    print("latest_ts:", latest_ts)
    print("profile:", profile_id, "freq:", freq, "qt:", qt)
    print("long:")
    print(out[out["side"] == "long"])
    print("short:")
    print(out[out["side"] == "short"])
    print("all groups:")
    print(out)

    return out

latest_group_df = print_latest_groups(result, fm, profile_id="perp_1d")

latest_ts: 2026-05-01 00:00:00
profile: perp_1d freq: 1D qt: 0.2
long:
                     factor  rank_pct    group  side
VELODROMEUSDT      2.331043  1.000000  group_9  long
BNTUSDT            2.155178  0.998088  group_9  long
AVAUSDT            2.076001  0.996176  group_9  long
1000000CHEEMSUSDT  2.068943  0.994264  group_9  long
BMTUSDT            2.000077  0.992352  group_9  long
...                     ...       ...      ...   ...
CVXUSDT            0.687351  0.808795  group_8  long
ARKUSDT            0.681856  0.806883  group_8  long
FLRUSDT            0.681284  0.804971  group_8  long
DOLOUSDT           0.672090  0.803059  group_8  long
DEGENUSDT          0.669377  0.801147  group_8  long

[105 rows x 4 columns]
short:
                 factor  rank_pct    group   side
OPNUSDT       -0.817351  0.198853  group_1  short
UAIUSDT       -0.817385  0.196941  group_1  short
FILUSDT       -0.819407  0.195029  group_1  short
BANANAS31USDT -0.821084  0.193117  group_1  short
DUSKUSDT    